# 5 Stufe 2: Zuordnung zur PN-AV

Die Detektionen aus Stufe 1 werden der provisorischen Numerisierung (PN-AV) zugeordnet und auf ganzen Plänen ausgewertet.
- Die PN bestimmt nur, **welche** Punkte gesucht sind, und liefert die Nummer. Die Lage kommt immer aus der Detektion.
- Alle PN-Punkte im Perimeter werden gesucht, auch unversicherte (verpasste Punkte sind schlimmer als Fehldetektionen).
- Jeder PN-Punkt bekommt höchstens eine Detektion im Umkreis `RADIUS_M` (eins zu eins). Bevorzugt wird die sicherste Detektion, der Abstand zur PN zählt nur schwach (die PN ist nur auf ca. 1 m genau).
- Detektionen ohne PN-Punkt in der Nähe werden nicht ausgegeben.
- Spalte `pruefen` markiert unsichere Punkte für die Kontrolle von Hand: `konfidenz` = Heatmap-Wert unter `SICHER`, `mehrdeutig` = weiterer Kandidat im Umkreis (z. B. altes und neues Symbol), `sicher` = kein Grund.

Ausgabe: `data/auswertung/stufe2_<version>/<plan>.csv` mit Punktnummer «a» + PN-Nummer, Lage (Pixel, LV95) und Markierung, dazu die Statistik.
Nur Pläne mit PN (Rüeggisberg, Wyssachen); für Hasle fehlt die PN-Zuordnung noch.

In [ ]:
import os
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from scipy.optimize import linear_sum_assignment

from grenzpunkte import DATENSATZ, MAX_ABSTAND, ZIEL_DPI, fenster_lesen

VERSION = "v1"  # "v1" = unet.pt, "v2" = unet_v2.pt (aus v2_training.ipynb)
DPI = None  # z. B. 300: Pläne bei der Inferenz als schlechter aufgelösten Scan simulieren
SPLIT = "val"
RADIUS_M = 3.0  # Suchradius um einen PN-Punkt in Metern (PN ist auf ca. 1 m genau)
SCHWELLE = 0.5  # Mindestwert der Heatmap für eine Detektion (tief, weil verpasste Punkte schlimmer sind)
SICHER = 0.9  # darunter wird ein Punkt zur Kontrolle markiert
AUSGABE = f"data/auswertung/stufe2_{VERSION}" + (f"_{DPI}dpi" if DPI else "")
os.makedirs(AUSGABE, exist_ok=True)

## Detektor wählen
V1 ohne DPI-Simulation läuft über `grenzpunkte.py`. Für V2 oder die 300-DPI-Simulation wird `grenzpunkte_v2.py` verwendet; es rechnet auch das V1-Modell.

In [ ]:
if VERSION == "v1" and DPI is None:
    from grenzpunkte import modell_laden, plan_detektieren
    model = modell_laden()
    detektieren = lambda plan_id, scan_dpi: plan_detektieren(model, plan_id, scan_dpi, SCHWELLE)
else:
    import grenzpunkte_v2
    model = grenzpunkte_v2.modell_laden(VERSION)
    detektieren = lambda plan_id, scan_dpi: grenzpunkte_v2.plan_detektieren(model, plan_id, scan_dpi, SCHWELLE, DPI)

## Zuordnung zur PN
Kosten = Abstand in Suchradien minus Heatmap-Wert: Eine sichere Detektion gewinnt vor einer näheren unsicheren. Paare weiter als `RADIUS_M` sind verboten.

In [ ]:
def pn_zuordnen(pn, detektionen, meter_pro_px):
    """Gibt die Paare (Index PN-Punkt, Index Detektion) und die Abstände in Metern zurück."""
    abstand_m = np.linalg.norm(pn[:, None, :] - detektionen[None, :, :2], axis=2) * meter_pro_px
    if len(pn) == 0 or len(detektionen) == 0:
        return [], abstand_m
    kosten = np.where(abstand_m <= RADIUS_M, abstand_m / RADIUS_M - detektionen[None, :, 2], 1e6)
    p, d = linear_sum_assignment(kosten)
    return [(i, j) for i, j in zip(p, d) if abstand_m[i, j] <= RADIUS_M], abstand_m


def markierung(i, j, paare, detektionen, abstand_m):
    """Grund, warum der PN-Punkt i mit Detektion j von Hand geprüft werden sollte ("sicher" = kein Grund)."""
    gruende = []
    if detektionen[j, 2] < SICHER:
        gruende.append("konfidenz")
    frei = np.ones(len(detektionen), bool)
    frei[[d for _, d in paare]] = False  # Detektionen, die keinem PN-Punkt zugeordnet sind
    if (frei & (abstand_m[i] <= RADIUS_M)).any():
        gruende.append("mehrdeutig")
    return "+".join(gruende) or "sicher"

## Alle Pläne: Stufe 1, Zuordnung und Vergleich mit den Labels
Pro Plan etwa 2–3 Minuten. Status pro PN-Punkt: `treffer`, `falsches_symbol` (Ausgabe mehr als 8 px vom Label), `fehldetektion` (ausgegeben, aber nicht digitalisiert), `verpasst`, `richtig_leer`.

In [ ]:
plaene = pd.read_csv(f"{DATENSATZ}/plaene.csv").query("status == 'ok'").set_index("plan_id")
split = pd.read_csv(f"{DATENSATZ}/split.csv").set_index("plan_id")["split"]
plan_ids = [p for p in split.index
            if split[p] == SPLIT and os.path.exists(f"{DATENSATZ}/helmert/pn_im_perimeter/{p}.csv")]

ergebnisse = []  # eine Zeile pro PN-Punkt
for plan_id in plan_ids:
    plan = plaene.loc[plan_id]
    faktor = ZIEL_DPI / plan["scan_dpi"]
    detektionen = detektieren(plan_id, plan["scan_dpi"])
    pn = pd.read_csv(f"{DATENSATZ}/helmert/pn_im_perimeter/{plan_id}.csv", dtype={"pn_nr": str})
    paare, abstand_m = pn_zuordnen(pn[["x_px", "y_px"]].values, detektionen, plan["bodenpixel_m"])
    pruefen = [markierung(i, j, paare, detektionen, abstand_m) for i, j in paare]

    # Ausgabe: Nummer "a" + PN-Nummer, Lage aus der Detektion
    xy = detektionen[[j for _, j in paare], :2].reshape(-1, 2)
    with rasterio.open(f"{DATENSATZ}/geotiff/{plan_id}.tif") as geotiff:
        E, N = geotiff.transform * (xy[:, 0], xy[:, 1])
    gefunden = pn.iloc[[i for i, _ in paare]]
    pd.DataFrame({"punkt_nr": "a" + gefunden["pn_nr"].values, "x_px": xy[:, 0].round(2), "y_px": xy[:, 1].round(2),
                  "E": E.round(3), "N": N.round(3), "konfidenz": detektionen[[j for _, j in paare], 2].round(3),
                  "vermarkung": gefunden["vermarkung"].values, "pruefen": pruefen}).to_csv(
        f"{AUSGABE}/{plan_id}.csv", index=False)

    # Vergleich mit den Labels: welches Label gehört zu welchem PN-Punkt
    zuordnung = pd.read_csv(f"{DATENSATZ}/zuordnung/{plan_id}.csv", dtype={"pn_nr": str}).dropna(subset=["pn_nr"])
    label_von_pn = dict(zip(zuordnung["pn_nr"], zuordnung[["x_px", "y_px"]].values))
    ausgabe_von_pn = {pn["pn_nr"].iloc[i]: (detektionen[j, :2], grund) for (i, j), grund in zip(paare, pruefen)}
    for nr, vermarkung in zip(pn["pn_nr"], pn["vermarkung"]):
        label = label_von_pn.get(nr)
        det, grund = ausgabe_von_pn.get(nr, (None, None))
        fehler_cm = np.nan
        if label is None and det is None:
            status = "richtig_leer"
        elif label is None:
            status = "fehldetektion"
        elif det is None:
            status = "verpasst"
        else:
            abstand_px = np.linalg.norm(det - label)
            status = "treffer" if abstand_px * faktor <= MAX_ABSTAND else "falsches_symbol"
            fehler_cm = abstand_px * plan["bodenpixel_m"] * 100
        ergebnisse.append({"plan": plan_id, "pn_nr": nr, "vermarkung": vermarkung, "status": status,
                           "fehler_cm": fehler_cm, "pruefen": grund})

ergebnisse = pd.DataFrame(ergebnisse)
ergebnisse.to_csv(f"{AUSGABE}/ergebnisse_{SPLIT}.csv", index=False)

## Statistik
Precision = Treffer / (Treffer + Fehldetektion + falsches Symbol), Recall = Treffer / (Treffer + verpasst + falsches Symbol). R_103 (1:5000) ist separat ausgewiesen, weil das Modell keinen 1:5000-Plan im Training hatte.

In [ ]:
def kennzahlen(e):
    n = e["status"].value_counts()
    tp, fp, fn, falsch = (n.get(s, 0) for s in ["treffer", "fehldetektion", "verpasst", "falsches_symbol"])
    fehler_cm = e.loc[e["status"] == "treffer", "fehler_cm"]
    ausgegeben = e["pruefen"].notna()
    return {"Treffer": tp, "Fehl": fp, "Verpasst": fn, "Falsch": falsch,
            "Precision": tp / (tp + fp + falsch), "Recall": tp / (tp + fn + falsch),
            "Median cm": fehler_cm.median(), "p95 cm": fehler_cm.quantile(0.95), "<= 10 cm": (fehler_cm <= 10).mean(),
            "markiert": (e.loc[ausgegeben, "pruefen"] != "sicher").mean()}


gruppen = {p: ergebnisse[ergebnisse["plan"] == p] for p in plan_ids}
gruppen |= {"alle": ergebnisse, "ohne R_103": ergebnisse[ergebnisse["plan"] != "R_103"]}
pd.DataFrame({name: kennzahlen(e) for name, e in gruppen.items()}).T.round(3)

Wie gut fängt die Markierung die Fehler ab? Anteil markierter Ausgaben pro Status (ohne R_103).

In [ ]:
e = ergebnisse[(ergebnisse["plan"] != "R_103") & ergebnisse["pruefen"].notna()]
e.groupby("status")["pruefen"].apply(lambda m: f"{(m != 'sicher').sum()} von {len(m)}")

## Bildbeispiele
Ausschnitte aus dem ganzen Plan: rotes × = Ausgabe des Modells, blauer Kreis = PN-Punkt (ca. 1 m genau), grünes + = manuell geklickt.
1. Ausgaben an PN-Punkten, die nicht digitalisiert wurden (zufällige 36)
2. Ausgaben, die mehr als 8 px neben dem Label liegen (alle)

In [ ]:
ZOOM = 60  # Scanpixel um die Ausgabe (bei 600 DPI und 1:1000 etwa 2.5 m)
random.seed(0)


def beispiele_sammeln(status):
    """Sucht zu jedem PN-Punkt mit diesem Status (ohne R_103) die Ausgabe und die PN-Lage in Pixeln."""
    beispiele = []
    for _, e in ergebnisse[(ergebnisse["status"] == status) & (ergebnisse["plan"] != "R_103")].iterrows():
        ausgabe = pd.read_csv(f"{AUSGABE}/{e['plan']}.csv").set_index("punkt_nr")
        pn = pd.read_csv(f"{DATENSATZ}/helmert/pn_im_perimeter/{e['plan']}.csv", dtype={"pn_nr": str}).set_index("pn_nr")
        beispiele.append({"plan": e["plan"], "pn_nr": e["pn_nr"],
                          "x": ausgabe.at["a" + e["pn_nr"], "x_px"], "y": ausgabe.at["a" + e["pn_nr"], "y_px"],
                          "pn_x": pn.at[e["pn_nr"], "x_px"], "pn_y": pn.at[e["pn_nr"], "y_px"]})
    return beispiele


def galerie(beispiele, titel, datei):
    spalten = 6
    zeilen = int(np.ceil(len(beispiele) / spalten))
    fig, axes = plt.subplots(zeilen, spalten, figsize=(3 * spalten, 3.3 * zeilen), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, b in zip(axes.flat, beispiele):
        x0, y0 = int(b["x"]) - ZOOM, int(b["y"]) - ZOOM
        with rasterio.open(f"{DATENSATZ}/geotiff/{b['plan']}.tif") as geotiff:
            bild = fenster_lesen(geotiff, x0, y0, 2 * ZOOM)
        labels = pd.read_csv(f"{DATENSATZ}/labels/{b['plan']}.csv")
        ax.imshow(bild, extent=(x0, x0 + 2 * ZOOM, y0 + 2 * ZOOM, y0), interpolation="nearest")
        ax.scatter(labels["x_px"], labels["y_px"], marker="+", c="lime", s=200, lw=1.5)
        ax.scatter(b["pn_x"], b["pn_y"], s=200, facecolors="none", edgecolors="blue", lw=1.5)
        ax.scatter(b["x"], b["y"], marker="x", c="red", s=150, lw=1.5)
        ax.set_xlim(x0, x0 + 2 * ZOOM)
        ax.set_ylim(y0 + 2 * ZOOM, y0)
        ax.set_title(f"{b['plan']}  PN {b['pn_nr']}", fontsize=8)
    fig.suptitle(titel)
    fig.tight_layout()
    fig.savefig(datei, dpi=100)
    plt.show()

In [ ]:
fehl = beispiele_sammeln("fehldetektion")
galerie(random.sample(fehl, min(36, len(fehl))), f"Ausgabe an nicht digitalisierten PN-Punkten ({len(fehl)} insgesamt)",
        f"{AUSGABE}/fehldetektionen.png")
falsch = beispiele_sammeln("falsches_symbol")
galerie(falsch, f"Falsches Symbol (alle {len(falsch)})", f"{AUSGABE}/falsches_symbol.png")